# JETANK Manual Control Panel

This notebook is based on the official `JETANK_1_servos` and `JETANK_2_ctrl` examples in this repository. It provides:

- Low-speed, small-angle jog controls for arm servos 1–4;
- A separate section for camera tilt servo 5;
- Forward, backward, left, right, and stop controls for the tracked base, with an adjustable speed slider.

> **Safety:** Raise the tracks off the ground or give the robot plenty of clear space first. A base direction button keeps the robot moving until you press **Stop Base** or **STOP ALL**. Servos 2 and 3 jointly determine the arm pose, so jogging them independently can still cause mechanical interference. Start with 1–2° steps at low speed.

> **Angle-state note:** `servoAngleCtrl()` uses an absolute target angle, but the official high-level examples do not expose current-position reading. The target angles shown below are software-tracked values initialized from `INITIAL_ANGLES`. If the robot is not currently at those angles, update the configuration before running or calibrate cautiously with very small steps. Running the notebook does not move any servo automatically.

In [ ]:
import atexit
import time

import ipywidgets as widgets
from IPython.display import display
from jetbot import Robot
from SCSCtrl import TTLServo

# Stop immediately after creating the base object so no motor state is inherited from another notebook.
robot = Robot()
robot.stop()
print('JETANK control libraries loaded. The base is stopped.')

## Adjustable configuration

The default ranges for servos 1, 4, and 5 come from the official examples. Conservative ranges are used for servos 2 and 3, but their actual safe ranges still depend on the mechanical assembly and current arm pose. Expand these limits only after checking that the mechanism cannot collide.

In [ ]:
SERVO_CONFIG = {
    1: {'name': 'Base / horizontal rotation', 'min': -80, 'max': 80},
    2: {'name': 'Arm shoulder joint', 'min': -90, 'max': 90},
    3: {'name': 'Arm elbow joint', 'min': -90, 'max': 90},
    4: {'name': 'Gripper open / close', 'min': -90, 'max': 0},
    5: {'name': 'Camera tilt', 'min': -45, 'max': 25},
}

# These values must match the robot's actual pose when jogging begins. Running this cell sends no servo command.
INITIAL_ANGLES = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}

# Wait briefly after each jog command before calling servoStop(). For small steps, 0.20 s is normally enough.
# Increase this value if a servo frequently stops before reaching its target.
DEFAULT_SERVO_STEP = 2
DEFAULT_SERVO_SPEED = 120
DEFAULT_STOP_DELAY = 0.20

## Create the control panel

Moving a parameter slider does not move the robot. Each click on a servo `−` / `+` button moves one configured step and then calls `servoStop()`. A base direction command remains active until you explicitly stop it.

In [ ]:
# If this cell is run again, stop any hardware controlled by the previous panel first.
try:
    robot.stop()
    for _sid in range(1, 6):
        TTLServo.servoStop(_sid)
        time.sleep(0.002)
except Exception:
    pass

servo_step = widgets.IntSlider(
    value=DEFAULT_SERVO_STEP, min=1, max=10, step=1,
    description='Step (°)', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='360px')
)
servo_speed = widgets.IntSlider(
    value=DEFAULT_SERVO_SPEED, min=20, max=500, step=10,
    description='Servo speed', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='360px')
)
stop_delay = widgets.FloatSlider(
    value=DEFAULT_STOP_DELAY, min=0.05, max=0.80, step=0.05,
    description='Stop delay (s)', readout_format='.2f', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='360px')
)
base_speed = widgets.FloatSlider(
    value=0.20, min=0.0, max=0.60, step=0.05,
    description='Base speed', readout_format='.2f', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='360px')
)

status = widgets.HTML(value='<b>Status:</b> Ready; the base is stopped.')
servo_angles = dict(INITIAL_ANGLES)
angle_labels = {}

button_layout = widgets.Layout(width='74px', height='38px')
wide_button_layout = widgets.Layout(width='130px', height='42px')

def set_status(message, color='#333'):
    status.value = f"<b>Status:</b> <span style='color:{color}'>{message}</span>"

def refresh_angle_label(servo_id):
    cfg = SERVO_CONFIG[servo_id]
    angle_labels[servo_id].value = (
        f"<b>{servo_angles[servo_id]:+d}°</b> "
        f"<span style='color:#777'>(limit {cfg['min']}…{cfg['max']}°)</span>"
    )

def stop_servo(servo_id, quiet=False):
    try:
        TTLServo.servoStop(servo_id)
        if not quiet:
            set_status(f'Servo {servo_id} stopped.')
    except Exception as exc:
        set_status(f'Failed to stop servo {servo_id}: {exc}', '#b00020')

def jog_servo(servo_id, direction):
    cfg = SERVO_CONFIG[servo_id]
    old_angle = servo_angles[servo_id]
    target = old_angle + direction * int(servo_step.value)
    target = max(cfg['min'], min(cfg['max'], target))

    if target == old_angle:
        set_status(f'Servo {servo_id} is already at the software limit {target:+d}°.', '#a05a00')
        return

    try:
        TTLServo.servoAngleCtrl(servo_id, target, 1, int(servo_speed.value))
        time.sleep(float(stop_delay.value))
        TTLServo.servoStop(servo_id)
        servo_angles[servo_id] = target
        refresh_angle_label(servo_id)
        set_status(f'Servo {servo_id} jogged to target {target:+d}° and was then stopped.', '#176b2c')
    except Exception as exc:
        try:
            TTLServo.servoStop(servo_id)
        except Exception:
            pass
        set_status(f'Servo {servo_id} control failed: {exc}', '#b00020')

def make_servo_row(servo_id):
    cfg = SERVO_CONFIG[servo_id]
    title = widgets.HTML(
        value=f"<b>Servo {servo_id}</b> — {cfg['name']}",
        layout=widgets.Layout(width='210px')
    )
    minus = widgets.Button(description='−', tooltip='Decrease target angle', layout=button_layout)
    plus = widgets.Button(description='+', tooltip='Increase target angle', layout=button_layout)
    stop = widgets.Button(description='Stop', button_style='warning', layout=button_layout)
    angle_labels[servo_id] = widgets.HTML(layout=widgets.Layout(width='220px'))
    refresh_angle_label(servo_id)

    minus.on_click(lambda _, sid=servo_id: jog_servo(sid, -1))
    plus.on_click(lambda _, sid=servo_id: jog_servo(sid, +1))
    stop.on_click(lambda _, sid=servo_id: stop_servo(sid))

    return widgets.HBox(
        [title, minus, plus, stop, angle_labels[servo_id]],
        layout=widgets.Layout(align_items='center', margin='3px 0')
    )

def drive(action):
    speed = float(base_speed.value)
    try:
        if action == 'stop':
            robot.stop()
            set_status('The base is stopped.', '#176b2c')
            return
        if speed <= 0:
            robot.stop()
            set_status('Base speed is 0; no movement command was sent.', '#a05a00')
            return
        {
            'forward': robot.forward,
            'backward': robot.backward,
            'left': robot.left,
            'right': robot.right,
        }[action](speed)
        names = {'forward': 'forward', 'backward': 'backward', 'left': 'left', 'right': 'right'}
        set_status(f"The base is moving {names[action]} at speed {speed:.2f}; press Stop to finish.", '#b04a00')
    except Exception as exc:
        try:
            robot.stop()
        except Exception:
            pass
        set_status(f'Base control failed: {exc}', '#b00020')

def stop_all(_=None, quiet=False):
    errors = []
    try:
        robot.stop()
    except Exception as exc:
        errors.append(f'base: {exc}')
    for servo_id in range(1, 6):
        try:
            TTLServo.servoStop(servo_id)
            time.sleep(0.002)
        except Exception as exc:
            errors.append(f'Servo {servo_id}: {exc}')
    if not quiet:
        if errors:
            set_status('Some stop commands failed: ' + '; '.join(errors), '#b00020')
        else:
            set_status('Stop commands were sent to the base and servos 1–5.', '#176b2c')

arm_rows = [make_servo_row(sid) for sid in (1, 2, 3, 4)]
camera_row = make_servo_row(5)

forward_button = widgets.Button(description='↑ Forward', layout=wide_button_layout)
backward_button = widgets.Button(description='↓ Backward', layout=wide_button_layout)
left_button = widgets.Button(description='← Left', layout=wide_button_layout)
right_button = widgets.Button(description='Right →', layout=wide_button_layout)
base_stop_button = widgets.Button(description='Stop Base', button_style='danger', layout=wide_button_layout)
all_stop_button = widgets.Button(
    description='STOP ALL', button_style='danger',
    layout=widgets.Layout(width='210px', height='48px')
)

forward_button.on_click(lambda _: drive('forward'))
backward_button.on_click(lambda _: drive('backward'))
left_button.on_click(lambda _: drive('left'))
right_button.on_click(lambda _: drive('right'))
base_stop_button.on_click(lambda _: drive('stop'))
all_stop_button.on_click(stop_all)

base_controls = widgets.VBox([
    widgets.HBox([forward_button], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([left_button, base_stop_button, right_button], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([backward_button], layout=widgets.Layout(justify_content='center')),
])

panel = widgets.VBox([
    widgets.HTML('<h3>Servo Jog Settings</h3>'),
    widgets.HBox([widgets.VBox([servo_step, servo_speed]), widgets.VBox([stop_delay])]),
    widgets.HTML('<h3>Arm Servos (1–4)</h3>'),
    widgets.VBox(arm_rows),
    widgets.HTML('<h3>Camera Tilt (Servo 5)</h3>'),
    camera_row,
    widgets.HTML('<h3>Tracked Base</h3>'),
    base_speed,
    base_controls,
    widgets.HBox([all_stop_button], layout=widgets.Layout(justify_content='center', margin='12px 0')),
    status,
], layout=widgets.Layout(border='1px solid #bbb', padding='12px', width='850px'))

# Try to stop the hardware when the kernel exits normally. Still run the final cleanup cell manually.
atexit.register(lambda: stop_all(quiet=True))
display(panel)

## Shutdown and cleanup

Run the cell below before finishing. It stops the base and all five servos. This notebook does not open the camera stream, so `camera.stop()` is not required.

In [ ]:
stop_all()
print('Cleanup complete: stop commands were sent to the base and all servos.')